In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# 機体点群の間引き
+ SCX2000向けに衝突判定に用いられる機体点群を除去する
+ lightningを作るためのプログラム一覧

In [ ]:
# notebookの実行パスからプロジェクトのルートへの相対パス
path_to_root = "../"

## ライブラリ

In [ ]:
import os

import numpy as np
import pandas as pd
import k3d

from argus_synchro.experiments.debug_vis.viewer_3d import create_simple_k3d_points

In [ ]:
import argus_synchro.SubScrutinizer as SubScrt

## 定数

In [ ]:
from configparser import ConfigParser, ExtendedInterpolation

from argus_synchro import shared_app_config
from argus_synchro.config.app_config import AppConfig
# import shared_app_config

In [ ]:
app_ini = ConfigParser(interpolation=ExtendedInterpolation())
app_ini.read(f"{path_to_root}/config/settings.ini", "UTF-8")
# 共有メモリに反映
app_config = AppConfig(app_ini)
# sac = shared_app_config.SharedAppConfig()
# app_config = sac.read()

## 目標

In [ ]:
(
    l_machine_col_weighted,
    machine_mobile_points_measure_weighted,
    machine_immobile_points_measure_weighted,
) = SubScrt.create_machine_points(
    f"{path_to_root}/config/crane3d/collision_detection/SCX2000-3/weighted",
    app_config.LiDARPosition,
    "col_machine_info.jsonc",
)

In [ ]:
machine_mobile_points_measure_weighted.shape, machine_immobile_points_measure_weighted.shape

+ 現状が50000点くらいなので、10分の1にすれば良さそう

# 機体の各部位で色分けて表示

In [ ]:
import k3d
import numpy as np
import matplotlib.pyplot as plt

from argus_synchro.experiments.debug_vis.viewer_3d import create_simple_k3d_points

In [ ]:
def colors_from_colormap_discrete(name: str, n: int):
    """matplotlib のカラーマップを n 等分サンプリングし、0xRRGGBB のリストを返す。"""
    cmap = plt.get_cmap(name)
    xs = np.linspace(0, 1, n)
    out = []
    for x in xs:
        r, g, b, _ = cmap(x)
        R, G, B = int(round(r*255)), int(round(g*255)), int(round(b*255))
        out.append((R << 16) | (G << 8) | B)
    return out

In [ ]:
palette = colors_from_colormap_discrete('tab20', len(l_machine_col_weighted))

plot = k3d.plot()
for parts, color in zip(l_machine_col_weighted, palette):
    plot += create_simple_k3d_points(parts.machine_pcd_points, color=color, point_size=0.05)

plot.display()

# V1のデータ間引き結果を格納する

## 定数の設定

In [ ]:
result_dir = f"{path_to_root}/config/crane3d/collision_detection/SCX2000-3/test_v1"
os.makedirs(result_dir, exist_ok=True)

## CW以外の上部旋回体
1. CW以外の上部旋回体の機体点群をDataFrameに詰め込む
2. 各機体部位に対して、select_imobile_machine_pointsを呼んで処理を行う
3. 結果のxyz軸を適切に反転させる
4. 結果をcsvに格納する

In [ ]:
from argus_synchro.experiments.machine_selection.SCX2000 import select_immobile_machine_points, groupby_argmin, groupby_argmax
# from experiments.machine_selection.SCX2000 import select_immobile_machine_points, groupby_argmin, groupby_argmax

In [ ]:
cell_size = [0.12, 0.12, 0.04]

In [ ]:
df_machine_immobile_without_cw = pd.concat((
    pd.DataFrame(
        parts.machine_pcd_points,
        columns=["x", "y", "z"])
    .assign(source = os.path.splitext(parts.pcd_points_file)[0])
    for parts in l_machine_col_weighted
    if "immobile" in parts.pcd_points_file and "CTWT" not in parts.pcd_points_file
), ignore_index=True)

In [ ]:
df_machine_immobile_without_cw.source.unique()

### キャットウォークの左部分

In [ ]:
df_catwalk_left = select_immobile_machine_points(
    df_machine_immobile_without_cw.
    query("source == 'immobile_pts/CATWALK_LEFT'")
    .reindex(columns=["x", "y", "z"]).values,
    cell_size=cell_size,
).fillna(True)\
.assign(
    source = 'immobile_pts/CATWALK_LEFT'
)

### キャットウォークの右部分

In [ ]:
df_catwalk_right = select_immobile_machine_points(
    df_machine_immobile_without_cw
    .query("source == 'immobile_pts/CATWALK_RIGHT'")
    .reindex(columns=["x", "y", "z"]).values,
    cell_size=cell_size,
).fillna(True)\
.assign(
    source = 'immobile_pts/CATWALK_RIGHT'
)

### キャブ部分

In [ ]:
df_cab = select_immobile_machine_points(
    df_machine_immobile_without_cw
    .query("source == 'immobile_pts/CAB'")
    .reindex(columns=["x", "y", "z"]).values,
    cell_size=cell_size,
    frac=[1.0, 1.0, 0.4],
    z_th=0.7,
)\
.assign(
    source = 'immobile_pts/CAB'
)

### house_l部分
+ データを作っているが、house_[lr]より先に他の部位が衝突しそうなので、機体点群としては空の点群を出力するようにしている

In [ ]:
df_house_l = select_immobile_machine_points(
    df_machine_immobile_without_cw.query("source == 'immobile_pts/HOUSE_L'").reindex(columns=["x", "y", "z"]).values,
    cell_size=cell_size,
    frac=[1.0, 0.2, 1.0],
    key_to_agg_y=["vox_x", "vox_z"],
).fillna(True)\
.assign(
    source = 'immobile_pts/HOUSE_L'
)

In [ ]:
plot = k3d.plot()

test_val = df_machine_immobile_without_cw.query("source == 'immobile_pts/HOUSE_L'").reindex(columns=["x", "y", "z"]).values
plot += create_simple_k3d_points(test_val)

test_val = df_machine_immobile_without_cw.query("source != 'immobile_pts/HOUSE_L'").reindex(columns=["x", "y", "z"]).values
plot += create_simple_k3d_points(test_val, color=0xff0000)

plot.display()


### house_r部分
+ データを作っているが、house_[lr]より先に他の部位が衝突しそうなので、機体点群としては空の点群を出力するようにしている

In [ ]:
df_house_r = select_immobile_machine_points(
    df_machine_immobile_without_cw.query("source == 'immobile_pts/HOUSE_R'").reindex(columns=["x", "y", "z"]).values,
    cell_size=cell_size,
    frac=[1.0, 0.2, 1.0],
    key_to_agg_y=["vox_x", "vox_z"],
).fillna(True)\
.assign(
    source = 'immobile_pts/HOUSE_R'
)

### メインフレーム部分

In [ ]:
df_mainframe = select_immobile_machine_points(
    df_machine_immobile_without_cw.query("source == 'immobile_pts/MAIN_FRAME'").reindex(columns=["x", "y", "z"]).values,
    cell_size=cell_size,
    frac=[0.5, 0.5, 0.5],
    key_to_agg_y = ["vox_x", "vox_z"],
    y_th=0.4,
    x_th=0.4,
)\
.fillna(True)\
.pipe(
    lambda df: pd.concat([
        groupby_argmin(df.query("is_min_y_in_vox").query("diff_min_z < 0.3"), ["vox_x", "vox_y"], "vox_z").assign(is_miny_in_vox_with_minz = True),
        df.assign(is_miny_in_vox_with_minz = False),
    ])\
    .drop_duplicates(["x", "y", "z"])
)\
.pipe(
    lambda df: pd.concat([
        groupby_argmin(df.query("is_max_y_in_vox").query("diff_min_z < 0.3"), ["vox_x", "vox_y"], "vox_z").assign(is_maxy_in_vox_with_minz = True),
        df.assign(is_maxy_in_vox_with_minz = False),
    ])\
    .drop_duplicates(["x", "y", "z"])
)\
.assign(
    source = 'immobile_pts/MAIN_FRAME'
)

In [ ]:
df_upper_parts = pd.concat([
    df_catwalk_right.query("is_max_y_in_vox or is_min_x_in_vox or is_max_x_in_vox").reindex(columns=["x", "y", "z", "source"]),
    df_catwalk_left.query("is_min_y_in_vox or is_min_x_in_vox or is_max_x_in_vox").reindex(columns=["x", "y", "z", "source"]),
    df_cab.query("is_min_z_in_vox").reindex(columns=["x", "y", "z", "source"]),
    df_house_l.query("is_min_y_in_vox").reindex(columns=["x", "y", "z", "source"]),
    df_house_r.query("is_max_y_in_vox").reindex(columns=["x", "y", "z", "source"]),
    df_mainframe.query("is_min_x_in_vox or is_max_x_in_vox or is_min_z_in_vox or is_miny_in_vox_with_minz or is_maxy_in_vox_with_minz").reindex(columns=["x", "y", "z", "source"])
])

### 結果の書き込み
+ house_[lr]は空の行列を書き込む

In [ ]:
for key, df_sub in df_upper_parts.groupby("source"):
    save_filename = f"{result_dir}/{key}.csv"
    
    os.makedirs(os.path.dirname(save_filename), exist_ok=True)
    saved_points=df_sub.reindex(columns=["x", "y", "z"]).values if key != "immobile_pts/HOUSE_L" and key != "immobile_pts/HOUSE_R" else np.empty((0,3))
    machine_conf = next(
        filter(
            lambda elem: os.path.splitext(elem.pcd_points_file)[0] == key, 
            l_machine_col_weighted
        )
    )
    # csvとして保持するデータはx,y,z座標が反転したりしているので、逆変換する
    for elem in range(3):
        if machine_conf.reverse_initial[elem]:
            saved_points[:,elem] = -1 * saved_points[:,elem] - machine_conf.offsets_initial[elem]
        else:
            saved_points[:,elem] = saved_points[:,elem] - machine_conf.offsets_initial[elem]
    
    np.savetxt(save_filename, saved_points, delimiter=" ",)

## カウンタウェイトの間引き
1. 200t機のカウンタウェイトは左, 右, 真ん中で形状が異なるので、それらを分ける
2. 分けたそれぞれで機体点群を選ぶ
3. 結果を統合してxyz軸を適切に反転させる
4. 最終的な結果をcsvに格納する

In [ ]:
from argus_synchro.experiments.machine_selection.SCX2000 import select_cw_v2
from argus_synchro.common.common import is_in_interval

# from experiments.machine_selection.SCX2000 import select_cw_v2
# from common.common import is_in_interval

In [ ]:
vox_size = [0.06, 0.06, 0.02]

In [ ]:
machine_immobile_cw = np.vstack([
    parts.machine_pcd_points
    for parts in l_machine_col_weighted
    if "immobile" in parts.pcd_points_file and "CTWT" in parts.pcd_points_file
])

In [ ]:
machine_immobile_cw_left = machine_immobile_cw[is_in_interval(machine_immobile_cw, y_range=(-1000, -1.51))]
machine_immobile_cw_right = machine_immobile_cw[is_in_interval(machine_immobile_cw, y_range=(1.51, 1000))]
machine_immobile_cw_center = machine_immobile_cw[is_in_interval(machine_immobile_cw, y_range=(-1.51, 1.51))]

In [ ]:
df_cw_left = select_cw_v2(
    machine_immobile_cw_left,
    np.quantile(machine_immobile_cw_left[:,2], [0.15, 0.51]),
    np.array([machine_immobile_cw_left[:,0].mean(), -1.51]),
    min_z_th = 0.3,
    cell_size=vox_size,
    frac=0.4
)\
.fillna({
    "on_band_near_max_r": False,
    "is_min_theta": False,
    "is_max_theta": False,
    "is_under": False,
})

In [ ]:
df_cw_right = select_cw_v2(
    machine_immobile_cw_right,
    np.quantile(machine_immobile_cw_right[:,2], [0.15, 0.51]),
    np.array([machine_immobile_cw_right[:,0].mean(), 1.51]),
    min_z_th = 0.3,
    cell_size=vox_size,
    frac=0.4
)\
.fillna({
    "on_band_near_max_r": False,
    "is_min_theta": False,
    "is_max_theta": False,
    "is_under": False,
})

In [ ]:
df_cw_center = select_cw_v2(
    machine_immobile_cw_center,
    np.quantile(machine_immobile_cw_center[:,2], [0.9]),
    np.array([machine_immobile_cw_center[:,0].mean(), 0]),
    min_z_th = 0.025,
    min_x_th = 0.3,
    max_x_th = 0.3,
    cell_size=vox_size,
    frac=[0.7, 1.0, 0.2],
)\
.fillna({
    "is_max_r_in_theta": False,
    "is_min_theta": False,
    "is_max_theta": False,
    "is_under": False,
})

In [ ]:
df_machine_cw = pd.concat([
    df_cw_left.assign(is_selected = lambda df: df.on_band_near_max_r | df.is_min_z).reindex(columns=["x", "y", "z", "is_selected"]),
    df_cw_center.assign(is_selected = lambda df: df.is_min_z | df.is_max_x).reindex(columns=["x", "y", "z", "is_selected"]),
    df_cw_right.assign(is_selected = lambda df: df.on_band_near_max_r | df.is_min_z).reindex(columns=["x", "y", "z", "is_selected"]),
])

In [ ]:
key = "immobile_pts/CTWT"
machine_conf = next(filter(lambda elem: os.path.splitext(elem.pcd_points_file)[0] == key, l_machine_col_weighted))
saved_points = df_machine_cw.query("is_selected").reindex(columns=["x", "y", "z"]).values

for elem in range(3):
    if machine_conf.reverse_initial[elem]:
        saved_points[:,elem] = -1 * saved_points[:,elem] - machine_conf.offsets_initial[elem]
    else:
        saved_points[:,elem] = saved_points[:,elem] - machine_conf.offsets_initial[elem]


In [ ]:
np.savetxt(
    f"{result_dir}/immobile_pts/CTWT.csv",
    saved_points,
    delimiter=" ",
)

## 下部走行体
1. 下部走行体の旋回中心周りの機体点群はひとまとめにする
2. CARBODY, JACK_UP系を一つにまとめて、衝突判定が起こりそうな側面部分だけselect_mobile_machine_points_in_boundaryで取り出す
3. 除外領域となる箇所にis_excludedというフラグを付ける
4. 統合した結果をばらして、xyz軸を適切に反転させる
5. 反転させる点群をcsvに格納する

In [ ]:
from argus_synchro.experiments.machine_selection.SCX2000 import select_mobile_machine_points_in_boundary
# from experiments.machine_selection.SCX2000 import select_mobile_machine_points_in_boundary

In [ ]:
cell_size=(0.24, 0.24, 0.08)

In [ ]:
df_machine_mobile_xyz = pd.concat((
    pd.DataFrame(parts.machine_pcd_points, columns=["x", "y", "z"]).assign(filename=os.path.splitext(parts.pcd_points_file)[0])
    for parts in l_machine_col_weighted
    if ("immobile" not in parts.pcd_points_file and "CONNECTOR" not in parts.pcd_points_file) and ("CARBODY" in parts.pcd_points_file or "JACK_UP" in parts.pcd_points_file)
))

In [ ]:
df_machine_mobile_center = select_mobile_machine_points_in_boundary(
    df_machine_mobile_xyz,
    cell_size,
    x_th=1.5,
    key_to_agg_x=["vox_y", "vox_z"],
    frac=[0.7, 1.0, 1.0],
)\
.pipe(
    # 除外領域となる箇所にis_excludedというフラグを付ける
    lambda df: pd.concat([
        df.query("~(0.6<x<1.6 and 2.2<y<3.2)")\
        .query("~(0.6<x<1.6 and -3.2<y<-2.2)")\
        .query("~(-2.0<x<-1.0 and -3.2<y<-2.2)")\
        .query("~(-2.2<x<-1.0 and 1.9<y<3.2)")\
        .assign(is_excluded = False),
        df.assign(is_excluded = True)
    ])\
    .drop_duplicates(["x", "y", "z"])
)

In [ ]:
for key, df_sub in df_machine_mobile_center.query("~is_excluded and (is_min_x_in_vox or is_max_x_in_vox)").groupby("filename"):
    saved_filename = f"{result_dir}/{key}.csv"
    os.makedirs(os.path.dirname(saved_filename), exist_ok=True)
    
    machine_conf = next(filter(lambda elem: os.path.splitext(elem.pcd_points_file)[0] == key, l_machine_col_weighted))
    saved_points=df_sub.reindex(columns=["x", "y", "z"]).values
    for elem in range(3):
        if machine_conf.reverse_initial[elem]:
            saved_points[:,elem] = -1 * saved_points[:,elem] - machine_conf.offsets_initial[elem]
        else:
            saved_points[:,elem] = saved_points[:,elem] - machine_conf.offsets_initial[elem]
    
    np.savetxt(
        saved_filename,
        saved_points,
        delimiter=" ",
    )
    # df_sub.to_csv(connect_path(result_dir, f"{key}.csv"), sep=" ")

## クローラーの間引き

In [ ]:
from argus_synchro.experiments.machine_selection.SCX2000 import select_mobile_machine_points_in_boundary, pipe_pcd_pd_to_np
# from experiments.machine_selection.SCX2000 import select_mobile_machine_points_in_boundary, pipe_pcd_pd_to_np

### クローラーの左部分の間引き
1. select_mobile_machine_in_boundaryで必要な箇所の点群を取り出す
2. クローラーのある高さの点群を線上に取り出す
3. xyz座標を適切に反転させる
4. 結果をcsvに格納する

In [ ]:
crawler_left_obj = next(filter(lambda elem: "CRAWLER_LEFT" in str(elem), l_machine_col_weighted))

In [ ]:
cell_size = (0.24, 0.24, 0.08)

z_th = 0.5
z_target = np.quantile(crawler_left_obj.machine_pcd_points[:,2], [0.5])
z_width = 0.2 / 2
except_y_th = 0.3

In [ ]:
df_crawler_left = select_mobile_machine_points_in_boundary(
    pd.DataFrame(crawler_left_obj.machine_pcd_points, columns=["x", "y", "z"]),
    cell_size=cell_size,
    z_th=z_th,
    key_to_agg_x=["vox_y", "vox_z"],
    key_to_agg_y=["vox_x", "vox_z"],
).fillna(True)\
.assign(
    diff_band = lambda df: np.abs(df.z - z_target[0]),
    vox_z_for_band = lambda df: (df.z // (z_width*4)).astype(int),
)\
.pipe(
    # 直線状の点を取り出す
    lambda df: pd.concat([
        groupby_argmin(
            df\
            .query(f"diff_band < {z_width}")\
            .query(f"diff_min_y > {except_y_th} and diff_max_y > {except_y_th}"), ["vox_x", "vox_y", "vox_z_for_band"], "diff_band").assign(on_band=True),
        df.assign(on_band=False),
    ]).drop_duplicates(["x", "y", "z"])
)

In [ ]:
select_cond = "on_band or is_max_z_in_vox or is_min_y_in_vox or is_min_x_in_vox or is_max_x_in_vox"
key = "mobile_pts/CRAWLER_LEFT"

In [ ]:
machine_conf = next(
    filter(
        lambda elem: os.path.splitext(elem.pcd_points_file)[0] == key, l_machine_col_weighted
    )
)
saved_points = df_crawler_left.query(select_cond).pipe(pipe_pcd_pd_to_np)

for elem in range(3):
    if machine_conf.reverse_initial[elem]:
        saved_points[:,elem] = -1 * saved_points[:,elem] - machine_conf.offsets_initial[elem]
    else:
        saved_points[:,elem] = saved_points[:,elem] - machine_conf.offsets_initial[elem]

In [ ]:
np.savetxt(
    f"{result_dir}/{crawler_left_obj}",
    saved_points,
    delimiter=" ",
)

### クローラーの右部分の間引き
1. select_mobile_machine_in_boundaryで必要な箇所の点群を取り出す
2. クローラーのある高さの点群を線上に取り出す
3. xyz座標を適切に反転させる
4. 結果をcsvに格納する

In [ ]:
crawler_right_obj = next(
    filter(
        lambda elem: "CRAWLER_RIGHT" in str(elem), l_machine_col_weighted
    )
)

In [ ]:
cell_size = (0.24, 0.24, 0.08)
z_th = 0.5
z_target = np.quantile(crawler_right_obj.machine_pcd_points[:,2], [0.5])
z_width = 0.2 / 2
except_y_th = 0.3

In [ ]:
df_crawler_right = select_mobile_machine_points_in_boundary(
    pd.DataFrame(crawler_right_obj.machine_pcd_points, columns=["x", "y", "z"]),
    cell_size=cell_size,
    z_th=z_th,
    key_to_agg_x=["vox_y", "vox_z"],
    key_to_agg_y=["vox_x", "vox_z"],
).fillna(True)\
.assign(
    diff_band = lambda df: np.abs(df.z - z_target[0]),
    vox_z_for_band = lambda df: (df.z // (z_width*4)).astype(int),
)\
.pipe(
    # 直線状の点を取り出す
    lambda df: pd.concat([
        groupby_argmin(
            df\
            .query(f"diff_band < {z_width}")\
            .query(f"diff_min_y > {except_y_th} and diff_max_y > {except_y_th}"), ["vox_x", "vox_y", "vox_z_for_band"], "diff_band").assign(on_band=True),
        df.assign(on_band=False),
    ]).drop_duplicates(["x", "y", "z"])
)

In [ ]:
key = "mobile_pts/CRAWLER_RIGHT"
machine_conf = next(
    filter(
        lambda elem: os.path.splitext(elem.pcd_points_file)[0] == key, l_machine_col_weighted
    )
)

In [ ]:
saved_points = df_crawler_right.query(select_cond).pipe(pipe_pcd_pd_to_np)

for elem in range(3):
    if machine_conf.reverse_initial[elem]:
        saved_points[:,elem] = -1 * saved_points[:,elem] - machine_conf.offsets_initial[elem]
    else:
        saved_points[:,elem] = saved_points[:,elem] - machine_conf.offsets_initial[elem]

In [ ]:
np.savetxt(
    f"{result_dir}/{crawler_right_obj}",
    saved_points,
    delimiter=" ",
)

# 間引いた結果の確認
1. 作った結果を置いたディレクトリにcol_machine_info.jsoncを置く
2. 機体除去点群のデータも置く
3. 読み込んで点数を確認

In [ ]:
(
    l_machine_col_confirm,
    _,
    _,
) = SubScrt.create_machine_points(
    result_dir,
    app_config.LiDARPosition,
    "col_machine_info.jsonc",
)

In [ ]:
{
    str(parts): len(parts.machine_pcd_points)
    for parts in l_machine_col_confirm
}

In [ ]:
{
    str(parts): len(parts.machine_pcd_points)
    for parts in l_machine_col_weighted
}

In [ ]:
plot = k3d.plot()

for parts in l_machine_col_confirm:
    plot += create_simple_k3d_points(parts.machine_pcd_points, point_size=0.05)

for parts in l_machine_col_weighted:
    plot += create_simple_k3d_points(parts.machine_pcd_points, color=0xff0000, point_size=0.01)

plot.display()